# KeyFlow Pipeline Verification & Comparison (Google Colab)

> **Step 1: Enable GPU Runtime**
>
> Before running any cells, ensure a GPU accelerator is connected:
> Navigate to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or any available GPU), then click **Save**.

In [ ]:
# Confirm GPU availability
!nvidia-smi

In [ ]:
# @title Clone KeyFlow Repository
# @markdown Enter the Git URL of your fork or repo. Note: edit `<user>` to match your GitHub account/repo.
repo_url = "https://github.com/<user>/Piano-Flow.git"  # @param {type:"string"}

import os

repo_dir = "/content/Piano-Flow"
if os.path.exists(repo_dir):
    print(f"Target directory {repo_dir} already exists. Pulling latest commits...")
    !cd {repo_dir} && git pull
else:
    print(f"Cloning {repo_url} into {repo_dir}...")
    !git clone {repo_url} {repo_dir}

%cd {repo_dir}

In [ ]:
# @title Alternative: Upload backend/ folder as a ZIP
# @markdown If you prefer not to clone from GitHub (e.g. uncommitted local changes), zip your local `backend/` folder and upload it here.
import os
import zipfile
from google.colab import files

print("Optional: Upload backend.zip (skip this cell if you cloned the repo in the previous cell)...")
uploaded_zips = files.upload()

for fname in uploaded_zips.keys():
    if fname.lower().endswith(".zip"):
        extract_target = "/content/Piano-Flow"
        os.makedirs(extract_target, exist_ok=True)
        with zipfile.ZipFile(fname, "r") as zf:
            zf.extractall(extract_target)
        print(f"Successfully extracted {fname} into {extract_target}!")
        %cd {extract_target}
        break

In [ ]:
# Install system audio dependencies and Python requirements
!apt-get update -qq && apt-get -y install ffmpeg libsndfile1
!pip install -r backend/requirements.txt

In [ ]:
# Check installed dependency versions to verify compatibility
import importlib

check_packages = [
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("torchaudio", "torchaudio"),
    ("audio-separator", "audio_separator"),
    ("transkun", "transkun"),
    ("demucs", "demucs"),
    ("piano_transcription_inference", "piano_transcription_inference"),
]

print("=== Installed Dependency Versions ===")
for display_name, mod_name in check_packages:
    try:
        m = importlib.import_module(mod_name)
        v = getattr(m, "__version__", "installed (no __version__)")
        print(f"  {display_name:<30} : {v}")
    except ImportError as err:
        print(f"  {display_name:<30} : NOT FOUND ({err})")

In [ ]:
# Upload a test song audio file
from google.colab import files
import os

print("Upload an audio file (e.g., .mp3, .wav, .m4a, .flac) to run through the pipelines:")
uploaded_audio = files.upload()

if not uploaded_audio:
    raise ValueError("No file uploaded. Please run this cell again and select an audio file.")

test_audio_path = list(uploaded_audio.keys())[0]
print(f"Ready: {test_audio_path} ({os.path.getsize(test_audio_path)} bytes)")

In [ ]:
# Run compare_pipelines.py across all pipeline combinations
# (Add --skip-demucs to test only the BS-RoFormer / Transkun pipeline)
!python backend/tools/compare_pipelines.py "{test_audio_path}"

In [ ]:
# Zip and download the comparison output folder
from pathlib import Path
import shutil
from google.colab import files

stem = Path(test_audio_path).stem
output_dir = Path(f"{stem}_compare")
zip_name = f"{stem}_compare.zip"

if output_dir.exists():
    print(f"Archiving {output_dir} -> {zip_name}...")
    shutil.make_archive(f"{stem}_compare", "zip", output_dir)
    print("Downloading comparison zip archive...")
    files.download(zip_name)
else:
    print(f"Warning: Output directory {output_dir} not found. Check previous cell output for errors.")

In [ ]:
# Start uvicorn server in background, probe /health, and test POST /api/transcribe
import subprocess
import time
import requests
import json
import os
from pathlib import Path

# Ensure port 8000 is clean
!fuser -k 8000/tcp || true

backend_dir = Path("/content/Piano-Flow/backend")
if not backend_dir.exists():
    backend_dir = Path("backend").resolve()

print("Starting uvicorn server:app on http://127.0.0.1:8000 in the background...")
server_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(backend_dir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

health_data = None
print("Probing /health...")
for attempt in range(20):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            health_data = r.json()
            print("Server /health is UP:")
            print(json.dumps(health_data, indent=2))
            break
    except Exception:
        pass

if not health_data:
    server_proc.terminate()
    out, _ = server_proc.communicate(timeout=5)
    raise RuntimeError(f"Server failed to start. Logs:\n{out}")

print(f"\nPOSTing '{test_audio_path}' to http://127.0.0.1:8000/api/transcribe ...")
try:
    with open(test_audio_path, "rb") as f:
        files_payload = {"file": (Path(test_audio_path).name, f, "audio/mpeg")}
        form_payload = {"isolate_piano": "true", "sensitivity": "0.5"}
        resp = requests.post(
            "http://127.0.0.1:8000/api/transcribe",
            files=files_payload,
            data=form_payload,
            timeout=300,
        )

    if resp.status_code == 200:
        res = resp.json()
        print("\n=== Transcription Succeeded ===")
        print(f"Title:            {res.get('title')}")
        print(f"Separator used:   {res.get('separator_used')}")
        print(f"Transcriber used: {res.get('transcriber_used')}")
        print(f"Timings:          {json.dumps(res.get('timings'), indent=2)}")
        print(f"MIDI Base64 len:  {len(res.get('midi_base64', ''))} chars")
    else:
        print(f"Transcription failed ({resp.status_code}): {resp.text}")
finally:
    print("\nStopping uvicorn server...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
    print("Server stopped.")